# 05 — GNN Experiments

**Phase 3 / Step 6 of the Homophily-Aware GNN for Fraud Ring Detection project.**

Graph neural networks on the temporal splits — the models the baselines
(Step 5) were the bar for. Models: GCN, GAT (4 heads), GraphSAGE (mean
aggregator), an H2GCN-style homophily-aware variant (ego embedding kept
separate, 1-hop + 2-hop neighbour channels), and a GatedGCN variant
(learned per-node gate blending ego and neighbour embeddings).

**Why the two homophily-aware variants (Step 4 evidence)**
- Illicit nodes link to illicit nodes at **5.9×** the random-mixing rate,
  so neighbour evidence is real — but node homophily is **bimodal** for
  illicit nodes (45% have zero same-label neighbours, 46% have all).
  Uniform smoothing (GCN) dilutes the ego signal of the zero-homophily
  illicit nodes.
- H2GCN-style keeps the ego embedding separate from neighbour
  aggregation and adds a 2-hop channel, so a node can rely on its own
  features when its neighbourhood is uninformative.
- GatedGCN learns a per-node gate from [ego, neighbour], so each node
  decides how much to trust its neighbourhood — directly addressing
  the bimodal case and licit nodes mixing broadly (0.72×).

**Setup (all constraints honoured)**
- **Splits are temporal** (train steps 1–34, val 35–41, test 42–49) —
  never random.
- **Supervised on labelled nodes only** (illicit vs licit). Unknown nodes
  (class 3) stay in the graph as message-passing context — never
  deleted, never training targets.
- **Features:** the identical pipeline to Step 5 — all 183 features,
  median imputation fit on TRAIN only + one missingness-indicator
  column, StandardScaler fit on train only.
- **Aggregation over INCOMING neighbours** (money senders), respecting
  the directed money-flow direction.
- **No SMOTE, no oversampling, no resampling** — class-weighted BCE
  loss only (pos_weight = licit/illicit on train).
- **Training:** full-batch, Adam lr 0.01, weight decay 5e-4, dropout
  0.5, hidden 64, 2 layers, up to 200 epochs, early stopping on val
  PR-AUC (patience 20). Model selection on VAL; final numbers on TEST.
- **Primary metric PR-AUC** (average precision). Accuracy is never
  headlined — only 2.23% of transactions are illicit.
- **Transductive:** all node features/edges are visible during message
  passing, but the loss uses train labels only; val labels only for
  early stopping/selection; test evaluated once at the end.

**Outputs**
- Figures → `../results/figures/gnn/` · reports →
  `../results/gnn_results.json`, `../results/gnn_summary.md`.

In [1]:
import sys
sys.path.insert(0, "../scripts")
import run_gnn as rg
import torch

print("torch", torch.__version__, "| threads:", torch.get_num_threads())
import torch_geometric
print("torch_geometric", torch_geometric.__version__)
print("models:", rg.MODELS)
print("XGBoost bar to beat (Step 5 test PR-AUC):", rg.XGB_TEST_PR_AUC)

torch 2.15.0+cpu | threads: 4
torch_geometric 2.8.1
models: ['gcn', 'gat', 'sage', 'h2gcn', 'gated_gcn']
XGBoost bar to beat (Step 5 test PR-AUC): 0.5652


/home/hatch/workspace/Homophily-Aware-GNN-Fraud-Ring-Detection/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
# 1. Load data — same feature pipeline as Step 5, plus directed edges
data = rg.load_graph_data()
print(f"nodes: {data['n_nodes']:,}, edges: {data['n_edges']:,}, "
      f"features: {len(data['feat_cols'])} + indicator = "
      f"{data['X'].shape[1]}")
for name in ("train", "val", "test"):
    m = data["masks"][name]
    print(f"{name:5s} labelled: {int(m.sum()):,} "
          f"(illicit {int((data['y'][m] == 1).sum()):,}, "
          f"licit {int((data['y'][m] == 0).sum()):,})")
print(f"pos_weight (licit/illicit train): {data['pos_weight']:.3f}")
print("edge_index shape:", data["edge_index"].shape,
      "| self-loops:", int((data['edge_index'][0]
                            == data['edge_index'][1]).sum()))

nodes: 203,769, edges: 234,355, features: 183 + indicator = 184
train labelled: 29,894 (illicit 3,462, licit 26,432)
val   labelled: 7,829 (illicit 675, licit 7,154)
test  labelled: 8,841 (illicit 408, licit 8,433)
pos_weight (licit/illicit train): 7.635
edge_index shape: (2, 234355) | self-loops: 0


/home/hatch/workspace/Homophily-Aware-GNN-Fraud-Ring-Detection/notebooks/../scripts/run_gnn.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ord = df.set_index("txId").loc[tx_ids].reset_index()


In [3]:
# 2. Torch tensors (CPU)
t = rg.to_tensors(data)
print("x:", tuple(t["x"].shape), t["x"].dtype)
print("edge_index:", tuple(t["edge_index"].shape))

x: (203769, 184) torch.float32
edge_index: (2, 234355)


In [4]:
# 3. Train all five GNNs (selection on val PR-AUC) — takes a while on CPU
results, fitted = rg.train_all(t, data["pos_weight"])
best = max(results, key=lambda n: results[n]["val"]["pr_auc"])
print(f"\nbest model by val PR-AUC: {best} "
      f"({results[best]['val']['pr_auc']:.4f})")

== training gcn ==
    [gcn] epoch   1 loss 2.2915 val PR-AUC 0.1774 (best 0.1774 @ epoch 1)
    [gcn] epoch  20 loss 0.5759 val PR-AUC 0.3409 (best 0.3409 @ epoch 20)
    [gcn] epoch  40 loss 0.4819 val PR-AUC 0.4242 (best 0.4242 @ epoch 40)
    [gcn] epoch  60 loss 0.4275 val PR-AUC 0.4985 (best 0.4985 @ epoch 60)
    [gcn] epoch  80 loss 0.3961 val PR-AUC 0.5368 (best 0.5426 @ epoch 78)
    [gcn] epoch 100 loss 0.3697 val PR-AUC 0.5531 (best 0.5710 @ epoch 95)
    [gcn] epoch 120 loss 0.3367 val PR-AUC 0.5647 (best 0.5762 @ epoch 119)
    [gcn] epoch 140 loss 0.3261 val PR-AUC 0.5693 (best 0.5944 @ epoch 130)
    [gcn] early stop at epoch 150 (best val PR-AUC 0.5944 @ epoch 130)
[gcn      ] best epoch 130 fit  171.7s  val PR-AUC=0.5944  test PR-AUC=0.2577
== training gat ==
    [gat] epoch   1 loss 3.0773 val PR-AUC 0.1312 (best 0.1312 @ epoch 1)
    [gat] epoch  20 loss 0.9322 val PR-AUC 0.4452 (best 0.4497 @ epoch 19)
    [gat] epoch  40 loss 0.8244 val PR-AUC 0.4508 (best 0.4708 

In [5]:
# 4. Results table — val and test, PR-AUC primary
print(f"{'model':9s} {'val PR-AUC':>10s} {'test PR-AUC':>11s} "
      f"{'test ROC-AUC':>12s} {'test F1@0.5':>11s} "
      f"{'prec@0.5':>9s} {'rec@0.5':>8s} {'fit s':>6s} {'epoch':>5s}")
for name in sorted(results, key=lambda n: results[n]["val"]["pr_auc"],
                   reverse=True):
    v, tt = results[name]["val"], results[name]["test"]
    print(f"{name:9s} {v['pr_auc']:10.4f} {tt['pr_auc']:11.4f} "
          f"{tt['roc_auc']:12.4f} {tt['f1_at_0.5']:11.4f} "
          f"{tt['precision_at_0.5']:9.4f} {tt['recall_at_0.5']:8.4f} "
          f"{results[name]['fit_seconds']:6.1f} "
          f"{results[name]['best_epoch']:5d}")
print(f"\nXGBoost bar (Step 5): test PR-AUC {rg.XGB_TEST_PR_AUC:.4f} — "
      f"best GNN beats it by "
      f"{results[best]['test']['pr_auc'] - rg.XGB_TEST_PR_AUC:+.4f}")
print("(accuracy is deliberately not reported — 2.23% illicit makes it "
      "meaningless here)")

model     val PR-AUC test PR-AUC test ROC-AUC test F1@0.5  prec@0.5  rec@0.5  fit s epoch
sage          0.7996      0.4746       0.8689      0.4631    0.4731   0.4534  303.6   140
h2gcn         0.7824      0.4102       0.8746      0.4235    0.3866   0.4681  953.1    86
gated_gcn     0.6881      0.3449       0.8656      0.3420    0.2599   0.5000  210.5    22
gcn           0.5944      0.2577       0.7943      0.3203    0.2547   0.4314  171.7   130
gat           0.4708      0.2502       0.8212      0.1724    0.0957   0.8701  133.4    36

XGBoost bar (Step 5): test PR-AUC 0.5652 — best GNN beats it by -0.0906
(accuracy is deliberately not reported — 2.23% illicit makes it meaningless here)


In [6]:
# 5. Fraud-ring evaluation (descriptive) — best model, test labelled nodes
fraud = rg.fraud_ring_eval(best, results, t, data)
print(f"predicted-illicit: {fraud['n_pred_illicit']:,} "
      f"(true {fraud['n_true_illicit']:,}), precision "
      f"{fraud['precision_illicit']:.4f}, recall "
      f"{fraud['recall_illicit']:.4f}")
print(f"pred subgraph: {fraud['pred_subgraph']['n_components']} weak "
      f"components, largest {fraud['pred_subgraph']['largest']}, "
      f"{fraud['pred_subgraph']['singletons']} singletons")
print(f"true subgraph: {fraud['true_subgraph']['n_components']} weak "
      f"components, largest {fraud['true_subgraph']['largest']}, "
      f"{fraud['true_subgraph']['singletons']} singletons")
print(f"pred illicit->illicit edge rate: "
      f"{fraud['pred_illicit_illicit_edge_rate']:.4f} "
      f"(true-illicit test rate "
      f"{fraud['true_illicit_illicit_edge_rate_test']:.4f}; "
      f"observed overall 0.2961)")
print("descriptive only — not proof of organised fraud rings")

predicted-illicit: 391 (true 408), precision 0.4731, recall 0.4534
pred subgraph: 330 weak components, largest 6, 290 singletons
true subgraph: 332 weak components, largest 12, 298 singletons
pred illicit->illicit edge rate: 0.2877 (true-illicit test rate 0.3838; observed overall 0.2961)
descriptive only — not proof of organised fraud rings


In [7]:
# 6. Explainability — GAT attention by edge type
attention = rg.gat_attention_analysis(fitted["gat"], t, data)
print(f"{'edge type':18s} {'n edges':>9s} {'mean att.':>10s} "
      f"{'median att.':>11s}")
for k, v in attention.items():
    print(f"{k:18s} {v['n_edges']:9,} {v['mean_attention']:10.4f} "
          f"{v['median_attention']:11.4f}")

edge type            n edges  mean att. median att.
illicit->illicit         998     0.4505      0.4965
illicit->licit           915     0.0653      0.0343
licit->illicit           781     0.2659      0.4328
licit->licit          33,930     0.2572      0.2241
unknown-involved     197,731     0.3412      0.4722


In [8]:
# 7. Feature-group ablation for the best model (retrains it twice)
ablation_res = rg.ablation(best, data, data["feat_cols"],
                           data["pos_weight"])
print(f"\n{'feature set':20s} {'val PR-AUC':>10s} {'test PR-AUC':>11s}")
for gname in ("local_only", "local_plus_aggregate"):
    a = ablation_res[gname]
    print(f"{gname:20s} {a['val']['pr_auc']:10.4f} "
          f"{a['test']['pr_auc']:11.4f}")
print(f"{'all features':20s} {results[best]['val']['pr_auc']:10.4f} "
      f"{results[best]['test']['pr_auc']:11.4f}")

== ablation local_only (94 features) ==
    [sage] epoch   1 loss 1.2658 val PR-AUC 0.2858 (best 0.2858 @ epoch 1)
    [sage] epoch  20 loss 0.4911 val PR-AUC 0.5615 (best 0.5615 @ epoch 20)
    [sage] epoch  40 loss 0.3739 val PR-AUC 0.6577 (best 0.6577 @ epoch 40)
    [sage] epoch  60 loss 0.3252 val PR-AUC 0.6884 (best 0.6899 @ epoch 59)
    [sage] epoch  80 loss 0.2950 val PR-AUC 0.7080 (best 0.7291 @ epoch 70)
    [sage] early stop at epoch 90 (best val PR-AUC 0.7291 @ epoch 70)
[local_only          ] val PR-AUC=0.7291  test PR-AUC=0.4143
== ablation local_plus_aggregate (166 features) ==
    [sage] epoch   1 loss 1.5379 val PR-AUC 0.4058 (best 0.4058 @ epoch 1)
    [sage] epoch  20 loss 0.3962 val PR-AUC 0.6134 (best 0.6143 @ epoch 19)
    [sage] epoch  40 loss 0.2849 val PR-AUC 0.6484 (best 0.6543 @ epoch 37)
    [sage] epoch  60 loss 0.2235 val PR-AUC 0.6988 (best 0.7092 @ epoch 59)
    [sage] epoch  80 loss 0.1927 val PR-AUC 0.7100 (best 0.7136 @ epoch 76)
    [sage] epoch 100

/home/hatch/workspace/Homophily-Aware-GNN-Fraud-Ring-Detection/notebooks/../scripts/run_gnn.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ord = df.set_index("txId").loc[tx_ids].reset_index()
/home/hatch/workspace/Homophily-Aware-GNN-Fraud-Ring-Detection/notebooks/../scripts/run_gnn.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ord = df.set_index("txId").loc[tx_ids].reset_index()


In [9]:
# 8. Figures (saved to ../results/figures/gnn/)
fig_paths = rg.make_figures(results, best, fraud, attention, ablation_res)
for fp in fig_paths:
    print("saved", fp.split("/")[-1])

saved gnn_pr_auc_bars.png
saved gnn_pr_curves.png
saved gnn_val_test_drop.png
saved gnn_training_curves.png
saved gnn_fraud_ring.png
saved gnn_gat_attention.png
saved gnn_ablation.png


In [10]:
# 9. Reports (JSON + Markdown summary)
rg.write_reports(results, best, fraud, attention, ablation_res, fig_paths)
print("wrote results/gnn_results.json, results/gnn_summary.md")
print(f"best model: {best} — val PR-AUC "
      f"{results[best]['val']['pr_auc']:.4f}, test PR-AUC "
      f"{results[best]['test']['pr_auc']:.4f}")

wrote results/gnn_results.json, results/gnn_summary.md
best model: sage — val PR-AUC 0.7996, test PR-AUC 0.4746


## Summary

- **Five GNNs trained on temporal splits** with class-weighted loss
  only — no SMOTE, no oversampling, no random splits.
- **Best model by val PR-AUC** and the full comparison table are in
  `results/gnn_summary.md`; per-model metrics in
  `results/gnn_results.json`.
- The XGBoost bar from Step 5 (test PR-AUC 0.5652) is the reference:
  the results table above shows whether, and by how much, the best
  GNN beats it.
- **Fraud-ring evaluation is descriptive** — component sizes and
  illicit→illicit edge rates of the predicted-illicit subgraph, not
  proof of organised rings.
- **Next:** Step 6 is complete — the full pipeline (EDA → graph →
  homophily → baselines → GNNs) is done. Remaining work is the final
  report update and any follow-up ablations.